# PHY224 — Notebook 01: Microscope stitching and CMOS geometric calibration

This notebook reconstructs the actual documented calibration workflow: SIFT / ratio test / RANSAC partial-affine registration for microscope views; seven-hole native-pixel red-channel diffraction landmark registration; and three-hole red-channel diffraction frame registration.

**Input archives (place alongside notebook or in `/mnt/data`):**
- `random_center_left.3.zip` (the 18 microscope views for 3-hole, 7-hole, and irregular patterns)
- `barch_2_0.zip` (7-hole Batch 2, reference and diffraction frames)
- `DAY_2_2.2cms With monitor.zip` (3-hole Day 2, Gaussian and baseline frames)
- `Krshna_1.zip` (irregular CMOS sequence; included for inventory, but its distance-to-filename mapping is not fully verified)

**Physical grid:** Native CMOS 2592 × 1944 pixels; pitch 2.775 µm. The SIFT working copies can be reduced **only for locating feature correspondences**, exactly as in the recovered microscope audit. Detector preprocessing and phase retrieval do **not** decimate pixels.

**Provenance:** The microscope SIFT/RANSAC code and seven-hole native landmark fit below are adapted directly from recovered source notebook / `.py` files bundled in `recovered_original_scripts/`. Adjustments are limited to configurable input paths and output folders. Three-hole native red-channel registration is a new, explicitly marked adapter; the earlier Day 2 script (included verbatim in the ZIP) operated on a 4× decimated RGB-magnitude representation.

In [ ]:
from pathlib import Path
import os, zipfile, itertools, csv, json, math
import cv2, numpy as np
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import gaussian_filter

SEARCH_DIRS = [Path.cwd(), Path('/mnt/data'), Path('/content')]
def find_input(name):
    for d in SEARCH_DIRS:
        p=d/name
        if p.is_file(): return p
    raise FileNotFoundError(f'{name} not found in {SEARCH_DIRS}; upload it beside notebook.')

BASE = Path.cwd()/'PHY224_outputs'
MIC_OUT=BASE/'microscopy'; MIC_RAW=MIC_OUT/'raw'
CAL_DIR=BASE/'calibration'; CMOS_DIR=CAL_DIR/'raw'
for p in [MIC_OUT,MIC_RAW,CAL_DIR,CMOS_DIR]:p.mkdir(parents=True,exist_ok=True)
NATIVE_SHAPE=(1944,2592)
PITCH_M=2.775e-6
WAVELENGTH_M=632.8e-9
print('Outputs:', BASE.resolve())

## A. Unpack microscope images (unmodified originals)

In [ ]:
with zipfile.ZipFile(find_input('random_center_left.3.zip')) as z:
    for name in z.namelist():
        if name.lower().endswith('.png'):
            (MIC_RAW/Path(name).name).write_bytes(z.read(name))
print('Unpacked microscope PNGs:',len(list(MIC_RAW.glob('*.png'))))

## B. Recovered microscope stitching: SIFT → ratio-test → RANSAC → connected mosaics

The recovered implementation permits disconnected components. Missing overlaps are **not** fabricated. Output mosaic raster is a preview; original microscope PNGs are preserved.

In [ ]:
import os, csv, itertools, math, json
import numpy as np, cv2
from PIL import Image,ImageDraw
from pathlib import Path
ROOT=MIC_RAW
OUT=MIC_OUT
GROUPS={
 'three_hole':['3_top.png','3_left.png','3_right.png','3_bottom.png'],
 'seven_hole':['7_top left.png','7_top right.png','7_center left.png','7_center right.png','7_bottom.png'],
 'random':['random_top_left.png','random_top.1_left.png','random_left.png','random_center_left.3.png','random_center.png','random_center.1.png','random_center.2.png','random_centerdown.png','random_bottom_left.png']
}
SCALE=0.45
sift=cv2.SIFT_create(nfeatures=2800,contrastThreshold=0.025)
matcher=cv2.BFMatcher(cv2.NORM_L2)
images={};features={}
for g, names in GROUPS.items():
 for name in names:
  im=cv2.imread(str(ROOT/name),cv2.IMREAD_GRAYSCALE)
  sm=cv2.resize(im,None,fx=SCALE,fy=SCALE,interpolation=cv2.INTER_AREA)
  kp,desc=sift.detectAndCompute(sm,None)
  images[name]=sm;features[name]=(kp,desc)
  print('feature',name,len(kp),flush=True)

def valid_homography(H):
 if H is None or not np.all(np.isfinite(H)): return False
 affine=H[:2,:2]; scales=np.linalg.svd(affine,compute_uv=False)
 det=np.linalg.det(affine)
 return 0.75<scales[0]<1.4 and 0.65<scales[1]<1.4 and det>0.45 and abs(H[2,0])<0.0015 and abs(H[2,1])<0.0015

pairs={};rows=[]
for group,names in GROUPS.items():
 for a,b in itertools.combinations(names,2):
  kp1,d1=features[a];kp2,d2=features[b]
  matches=matcher.knnMatch(d1,d2,k=2)
  good=[m for m,n in matches if m.distance < 0.7*n.distance]
  if len(good)<12: continue
  src=np.float32([kp1[m.queryIdx].pt for m in good]);dst=np.float32([kp2[m.trainIdx].pt for m in good]);
  H,mask=cv2.estimateAffinePartial2D(src,dst,method=cv2.RANSAC,ransacReprojThreshold=4,maxIters=2500,confidence=0.995)
  if H is None: continue
  H3=np.vstack([H,[0,0,1]])
  inlier=int(mask.sum()); ratio=inlier/len(good)
  scale=np.linalg.svd(H[:2,:2],compute_uv=False)[0]
  # Require high confidence in transformation
  if inlier < 14 or ratio < .29 or not (.79<scale<1.24): continue
  pairs[(a,b)]={'H':H3,'n':inlier,'ratio':ratio}
  pairs[(b,a)]={'H':np.linalg.inv(H3),'n':inlier,'ratio':ratio}
  rows.append({'group':group,'image_a':a,'image_b':b,'inliers':inlier,'matches':len(good),'inlier_fraction':ratio,'scale':scale,'tx':H[0,2],'ty':H[1,2]})
  print('match',group,a,b,inlier,len(good),round(ratio,2),round(scale,3),flush=True)

with open(OUT/'pair_matches.csv','w',newline='') as fp:
 w=csv.DictWriter(fp,fieldnames=['group','image_a','image_b','inliers','matches','inlier_fraction','scale','tx','ty']);w.writeheader();w.writerows(rows)

# Max confidence paths from all nodes to anchor through directed pair graph
import heapq
results={}
for group,names in GROUPS.items():
 anchor=names[len(names)//2]
 # choose root highest weighted degree (likely center overlap)
 scores={n:sum(v['n'] for (x,y),v in pairs.items() if x==n and y in names) for n in names}
 anchor=max(names,key=lambda name:scores[name]);print('ANCHOR',group,anchor,flush=True)
 # transforms H_a_to_anchor, start from root and expand
 transforms={anchor:np.eye(3)}
 reached=True
 while reached:
  reached=False;best=None
  for a in names:
   if a in transforms:continue
   for b in transforms:
    edge=pairs.get((a,b))
    if edge and (best is None or edge['n']*edge['ratio']>best[0]): best=(edge['n']*edge['ratio'],a,b,edge['H'])
  if best:
   _,a,b,H=best
   transforms[a]=transforms[b]@H;reached=True
   print('PLACE',a,'via',b,flush=True)
 missing=[n for n in names if n not in transforms]
 print('MISSING',group,missing,flush=True)
 # assemble at preview only
 H0,W0=next(iter(images.values())).shape
 corners=np.array([[0,0,1],[W0,0,1],[W0,H0,1],[0,H0,1]],float).T
 vertices=[]
 for n,H in transforms.items():
  v=H@corners; v=v[:2]/v[2]
  vertices.append(v)
 pts=np.concatenate(vertices,axis=1)
 lo=np.floor(np.min(pts,axis=1)).astype(int);hi=np.ceil(np.max(pts,axis=1)).astype(int)
 # limit potentially wrong placement
 if np.prod(hi-lo)>120_000_000:print('MOSAIC TOO LARGE',group,lo,hi,flush=True);continue
 bias=np.array([[1,0,-lo[0]+50],[0,1,-lo[1]+50],[0,0,1]])
 WW,HH=tuple((hi-lo+100).tolist())
 numerator=np.zeros((HH,WW),np.float32);denominator=np.zeros((HH,WW),np.float32)
 mask=np.full((H0,W0),1,np.float32)
 for n,H in transforms.items():
  tr=bias@H
  warped=cv2.warpPerspective(images[n].astype(np.float32),tr,(WW,HH),flags=cv2.INTER_LINEAR)
  weight=cv2.warpPerspective(mask,tr,(WW,HH),flags=cv2.INTER_LINEAR)
  numerator += warped*weight
  denominator += weight
 result=np.divide(numerator,denominator,out=np.zeros_like(numerator),where=denominator>0.05)
 # Scale to 1920 for preview
 f=min(1,1920/WW,1200/HH)
 if f<1: result=cv2.resize(result,None,fx=f,fy=f,interpolation=cv2.INTER_AREA)
 result=np.clip(result,0,255).astype(np.uint8)
 cv2.imwrite(str(OUT/f'{group}_mosaic.png'),result)
 results[group]={'anchor':anchor,'missing':missing,'covered':list(transforms),'pixel_scale_preview':SCALE,'transforms_to_anchor':{n:H.tolist() for n,H in transforms.items()},'mosaic_bounds_preview':{'width':WW,'height':HH,'left':int(lo[0]),'top':int(lo[1])}}
 print('saved',group,result.shape,flush=True)
with open(OUT/'stitching_metadata.json','w') as fp:json.dump(results,fp,indent=2)

## C. Native-resolution seven-hole CMOS landmark registration

This cell reproduces the native-pixel detector geometry fit used in Batch 2. It estimates an in-plane similarity transform from bright diffraction landmarks, not absolute camera yaw or an optical irradiance calibration.

In [ ]:
"""Geometric proxy registration for Batch-2 CMOS frames using ALL native pixels.
The Gaussian camera image moved: it is deliberately not force-registered by this script.
Matches broad diffraction-intensity landmarks, which evolve with z. Thus this is an
approximate in-plane transform, NOT a physical intensity/phase calibration.
"""
from pathlib import Path
import zipfile, csv
import numpy as np, cv2
from scipy.optimize import linear_sum_assignment
ROOT=Path(find_input('barch_2_0.zip')).parent
OUT=CAL_DIR
FILES={'mask':'batch_2_binary mask.bmp','z25':'battch2_2.5cms.bmp',
       'z31':'Batch2_3.1cms.bmp','z68':'batch2_6.8cms.bmp'}

def read_red(z,name):
    data=np.frombuffer(z.read(name),np.uint8)
    a=cv2.imdecode(data,cv2.IMREAD_COLOR)
    if a is None or a.shape[:2]!=(1944,2592):raise RuntimeError(name)
    return np.maximum(a[:,:,2].astype(np.float32)-1,0)

def landmarks(r):
    smooth=cv2.GaussianBlur(r,(0,0),14.0)
    bw=(smooth>max(19.,float(smooth.max())*.22)).astype(np.uint8)
    n,lab,stats,cent=cv2.connectedComponentsWithStats(bw,8)
    components=sorted([(stats[i,cv2.CC_STAT_AREA],cent[i])
                  for i in range(1,n) if stats[i,cv2.CC_STAT_AREA]>560],
                  key=lambda p:-p[0])[:7]
    del lab
    return np.array([c for _,c in components],np.float64)

def similarity(P,Q):
    pm=P.mean(axis=0);qm=Q.mean(axis=0)
    U,s,Vt=np.linalg.svd((P-pm).T@(Q-qm))
    R=U@Vt
    if np.linalg.det(R)<0:
        Vt[-1,:]*=-1;R=U@Vt
    a=np.sum(((P-pm)@R)*(Q-qm))/max(np.sum((P-pm)**2),1e-12)
    A=a*R
    t=qm-pm@A
    return A,t

def register(src,dst):
    A=np.eye(2);t=dst.mean(axis=0)-src.mean(axis=0)
    for _ in range(6):
        distance=np.linalg.norm(src[:,None,:]@A+t-dst[None,:,:],axis=-1)
        ri,ci=linear_sum_assignment(distance)
        A,t=similarity(src[ri],dst[ci])
    residual=np.linalg.norm(src[ri]@A+t-dst[ci],axis=1)
    affine=np.column_stack((A.T,t)).astype(np.float32)
    return affine,residual,src[ri]@A+t,len(ri)

rows=[]
with zipfile.ZipFile(ROOT/'barch_2_0.zip') as z:
    ref=landmarks(read_red(z,FILES['mask']))
    for name in ['z25','z31','z68']:
        pts=landmarks(read_red(z,FILES[name]))
        matrix,res,reg,n=register(pts,ref)
        out=OUT/f'native_registration_{name}.npz'
        np.savez_compressed(out,affine=matrix,registered_points_src=reg,reference_centers=ref)
        rows.append({'frame':name,'matched_landmarks':n,
           'rms_native_px':float(np.sqrt(np.mean(res**2))),
           'rms_um_assuming_pitch':float(np.sqrt(np.mean(res**2))*2.775),
           'translation_x_native_px':float(matrix[0,2]),
           'translation_y_native_px':float(matrix[1,2]),
           'scale':float(np.sqrt(np.linalg.det(matrix[:,:2]))),
           'rotation_deg':float(np.degrees(np.arctan2(matrix[1,0],matrix[0,0]))),
           'output':out.name})
with (OUT/'native_registration_quality.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
print(rows)

## D. Native-resolution three-hole CMOS calibration (red-channel adapter)

The Day 2 archive includes a long-distance baseline RGB photograph. Here RGB is subtracted pixelwise **before selecting red**, without downsampling. The original Day 2 RGB-magnitude implementation is preserved separately as `calibrated_day2.py`.

In [ ]:
DAY2_FILES = {
 'near':'DAY_2_WITH MONITOR BINARY CAMERA STABLE.bmp',
 'gauss1':'DAY_2_GAUSSIAN_WITH MONITOR_BINARY MASK CAMRA STABLE_1.bmp',
 'gauss2':'DAY_2_GAUSSIAN_WITH MONITOR_BINARY MASK CAMRA STABLE_2.bmp',
 'blank':'day_2_baseline_forall z more than 10.7cms.bmp',
 'z22':'DAY_2_2.2cms With monitor.bmp',
 'z34':'DAY_2_3.4cms withnitor.bmp',
 'z59':'DAY_2_5.9cms.bmp',
 'z107':'DAY_2_10.7cms.bmp',
 'z118':'DAY_2_11.8cms.bmp',
 'z245':'DAY_2_24.5cms.bmp'}
DAY2_CM = {'z22':2.2,'z34':3.4,'z59':5.9,'z107':10.7,'z118':11.8,'z245':24.5}

def bmp_from_zip(z, name):
    a=cv2.imdecode(np.frombuffer(z.read(name),dtype=np.uint8),cv2.IMREAD_COLOR)
    if a is None or a.shape[:2] != NATIVE_SHAPE:
        raise ValueError(f'Expected 2592×1944 BGR BMP: {name}; got {None if a is None else a.shape}')
    return a.astype(np.float32)

def subtract_rgb_then_red(image, baseline):
    # OpenCV BGR: index 2 is R. Subtract all three channel values FIRST.
    corrected=np.maximum(image.astype(np.float32)-baseline.astype(np.float32),0)
    return corrected[:,:,2]

with zipfile.ZipFile(find_input('DAY_2_2.2cms With monitor.zip')) as z:
    base_rgb=bmp_from_zip(z,DAY2_FILES['blank'])
    # This baseline was acquired for >10.7 cm. Applying it to other distances is
    # an explicit user-specified processing choice, not independently calibrated.
    raw_three={k:subtract_rgb_then_red(bmp_from_zip(z,n),base_rgb)
               for k,n in DAY2_FILES.items() if k!='blank'}

# Register the two Gaussian captures in camera coordinates (translation only).
g1=cv2.GaussianBlur(raw_three['gauss1'],(0,0),12)
g2=cv2.GaussianBlur(raw_three['gauss2'],(0,0),12)
shift,response=cv2.phaseCorrelate(g1,g2)
if np.hypot(*shift)>60 or response<.03:shift=(0.,0.)
M_g=np.array([[1,0,-shift[0]],[0,1,-shift[1]]],np.float32)
g2r=cv2.warpAffine(raw_three['gauss2'],M_g,(2592,1944))
beam=gaussian_filter((raw_three['gauss1']+g2r)/2,40).astype(np.float32)

def three_centres(arr):
    blur=cv2.GaussianBlur(arr,(0,0),14)
    m=(blur>max(19,float(blur.max())*.22)).astype(np.uint8)
    n,l,s,c=cv2.connectedComponentsWithStats(m,8)
    items=sorted([(int(s[i,4]),c[i]) for i in range(1,n) if s[i,4]>560],reverse=True,key=lambda x:x[0])[:3]
    if len(items)!=3:raise RuntimeError(f'Expected 3 diffraction landmarks, found {len(items)}')
    pts=np.asarray([xy for _,xy in items],np.float64)
    order=np.argsort(pts[:,1]); upper=pts[order[:2]][np.argsort(pts[order[:2],0])]
    return np.vstack([upper,pts[order[2]]]).astype(np.float32)

reference=three_centres(raw_three['near'])
registration=[]
for name in DAY2_CM:
    src=three_centres(raw_three[name])
    src0=src-src.mean(0); dst0=reference-reference.mean(0)
    U,_,Vt=np.linalg.svd(src0.T@dst0)
    R=U@Vt
    if np.linalg.det(R)<0: U[:,-1]*=-1; R=U@Vt
    A=np.zeros((2,3),np.float32)
    A[:,:2]=R.T
    A[:,2]=reference.mean(0)-src.mean(0)@R
    output=cv2.warpAffine(raw_three[name],A,(2592,1944),flags=cv2.INTER_LINEAR)
    valid=cv2.warpAffine(np.ones(NATIVE_SHAPE,np.uint8),A,(2592,1944),flags=cv2.INTER_NEAREST)>0
    pts=cv2.transform(src[None,:,:],A)[0]
    rms=float(np.sqrt(np.mean(np.sum((pts-reference)**2,axis=1))))
    np.savez_compressed(CAL_DIR/f'day2_registration_{name}.npz',red=output,valid=valid,
                        affine=A,reference_centers=reference,source_centers=src)
    registration.append({'frame':name,'z_cm':DAY2_CM[name], 'rms_native_px':rms,
                         'rotation_deg':float(np.degrees(np.arctan2(A[1,0],A[0,0]))),
                         'dx_native_px':float(A[0,2]),'dy_native_px':float(A[1,2])})
np.savez_compressed(CAL_DIR/'day2_red_gaussian_beam.npz',beam=beam, near_red=raw_three['near'])
with (CAL_DIR/'day2_native_red_registration.csv').open('w',newline='') as f:
    wr=csv.DictWriter(f,fieldnames=registration[0].keys());wr.writeheader();wr.writerows(registration)
print('Day2 Gaussian shift:',shift,'confidence:',response)
print('Native Day2 frames saved:',len(registration))
for r in registration:print(r)

## E. Check outputs

The registration `.npz` files and original source images are inputs to Notebook 02.

In [ ]:
print('Microscope mosaics:',[p.name for p in MIC_OUT.glob('*_mosaic.png')])
print('Seven-hole registrations:',[p.name for p in CAL_DIR.glob('native_registration_*.npz')])
print('Three-hole registrations:',[p.name for p in CAL_DIR.glob('day2_registration_*.npz')])
print('Outputs:', BASE.resolve())